<a href="https://colab.research.google.com/github/shivanjali26/NLP/blob/main/Assignment_8_nlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import numpy as np
import pandas as pd
import string
import nltk

from nltk.corpus import stopwords
from gensim.models import Word2Vec
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

nltk.download('stopwords')

data = {
    "query": [
        # Billing
        "I have an issue with my bill",
        "Payment failed but money deducted",
        "Why is my bill so high",
        "Incorrect charges in my bill",
        "Refund not processed",
        "Double payment charged",

        # Technical
        "Internet is not working",
        "WiFi connection is slow",
        "Technical issue with router",
        "Network problem persists",
        "Server is down",
        "App is crashing frequently",

        # General
        "How to reset my password",
        "Need help with account login",
        "How can I change my email",
        "Help me update profile",
        "Where can I see my account details",
        "How to contact support"
    ],
    "label": [
        "Billing","Billing","Billing","Billing","Billing","Billing",
        "Technical","Technical","Technical","Technical","Technical","Technical",
        "General","General","General","General","General","General"
    ]
}

df = pd.DataFrame(data)

# Preprocess
stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = text.lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    return [w for w in text.split() if w not in stop_words]

df["tokens"] = df["query"].apply(preprocess)

# Word2Vec
w2v_model = Word2Vec(df["tokens"], vector_size=100, window=5, min_count=1)

def sentence_vector(words):
    vecs = [w2v_model.wv[w] for w in words if w in w2v_model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(100)

X = np.array([sentence_vector(words) for words in df["tokens"]])
y = df["label"]

# Split (stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

# Logistic Regression
model = LogisticRegression()
model.fit(X_train, y_train)

# Evaluation
y_pred = model.predict(X_test)

# Test
def predict_query(text):
    words = preprocess(text)
    vec = sentence_vector(words)
    return model.predict([vec])[0]

print("\n===== SAMPLE OUTPUT =====")

queries = [
    "My bill is incorrect",
    "Internet is not working",
    "I forgot my password"
]

for q in queries:
    print("\nQuery:", q)
    print("Prediction:", predict_query(q))


===== SAMPLE OUTPUT =====

Query: My bill is incorrect
Prediction: Billing

Query: Internet is not working
Prediction: Technical

Query: I forgot my password
Prediction: General


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
